In [1]:
import polars as pl
import screed

In [2]:
df = pl.read_csv('bac120_metadata_r226.tsv.gz', separator='\t', ignore_errors=True)

In [3]:
df.head(1).columns

['accession',
 'ambiguous_bases',
 'checkm2_completeness',
 'checkm2_contamination',
 'checkm2_model',
 'checkm_completeness',
 'checkm_contamination',
 'checkm_marker_count',
 'checkm_marker_lineage',
 'checkm_marker_set_count',
 'checkm_strain_heterogeneity',
 'coding_bases',
 'coding_density',
 'contig_count',
 'gc_count',
 'gc_percentage',
 'genome_size',
 'gtdb_genome_representative',
 'gtdb_representative',
 'gtdb_taxonomy',
 'gtdb_type_designation_ncbi_taxa',
 'gtdb_type_designation_ncbi_taxa_sources',
 'gtdb_type_species_of_genus',
 'l50_contigs',
 'l50_scaffolds',
 'longest_contig',
 'longest_scaffold',
 'lsu_23s_contig_len',
 'lsu_23s_count',
 'lsu_23s_length',
 'lsu_23s_query_id',
 'lsu_5s_contig_len',
 'lsu_5s_count',
 'lsu_5s_length',
 'lsu_5s_query_id',
 'lsu_silva_23s_blast_align_len',
 'lsu_silva_23s_blast_bitscore',
 'lsu_silva_23s_blast_evalue',
 'lsu_silva_23s_blast_perc_identity',
 'lsu_silva_23s_blast_subject_id',
 'lsu_silva_23s_taxonomy',
 'mean_contig_length',
 

In [4]:
df.head(1).select([ 'gtdb_genome_representative',
 'gtdb_representative',
 'gtdb_taxonomy',])

gtdb_genome_representative,gtdb_representative,gtdb_taxonomy
str,str,str
"""RS_GCF_003697165.2""","""f""","""d__Bacteria;p__Pseudomonadota;…"


In [5]:
df.head(1)['gtdb_taxonomy'].item()

'd__Bacteria;p__Pseudomonadota;c__Gammaproteobacteria;o__Enterobacterales;f__Enterobacteriaceae;g__Escherichia;s__Escherichia coli'

In [6]:
CORE_NAMES=set([ x.strip() for x in open('inputs.branchwater/names.list') ])

In [7]:
df2 = df.filter(
    (pl.col('gtdb_taxonomy').str.split(';').list.get(-1).is_in(CORE_NAMES)) &
    (pl.col('gtdb_representative') == 't')
)

In [8]:
df3 = df2.with_columns(
    acc=pl.col('gtdb_genome_representative').str.slice(3)
)
df3.write_csv('PRJEB88111.nanopore/reps/reps.csv')

In [9]:
print("\n".join(df3['acc']))

GCA_900546925.1
GCA_022774325.1
GCF_002251295.1
GCF_021531895.1
GCA_004557565.1
GCF_009695905.1
GCA_002320005.1
GCF_002706375.1
GCA_902472425.1
GCA_002299675.1
GCA_034089285.1
GCF_019052365.1
GCF_009696175.1
GCA_004552595.1
GCA_022785155.1
GCA_000434935.1


## build list of contigs per species 

In [10]:
ls reps

ls: cannot access 'reps': No such file or directory


In [11]:
pwd


'/group/datalabgrp/ctbrown/2026-pancoding-core'

In [12]:
ls PRJEB88111.nanopore/reps/

GCA_000434935.1_genomic.fna.gz  GCF_002706375.1_genomic.fna.gz
GCA_002299675.1_genomic.fna.gz  GCF_009695905.1_genomic.fna.gz
GCA_002320005.1_genomic.fna.gz  GCF_009696175.1_genomic.fna.gz
GCA_004552595.1_genomic.fna.gz  GCF_019052365.1_genomic.fna.gz
GCA_004557565.1_genomic.fna.gz  GCF_021531895.1_genomic.fna.gz
GCA_022774325.1_genomic.fna.gz  all-core.fa
GCA_022785155.1_genomic.fna.gz  bams/
GCA_034089285.1_genomic.fna.gz  contigs_to_species.csv
GCA_900546925.1_genomic.fna.gz  list.txt
GCA_902472425.1_genomic.fna.gz  reps.csv
GCF_002251295.1_genomic.fna.gz


In [13]:
df3['acc']

acc
str
"""GCA_900546925.1"""
"""GCA_022774325.1"""
"""GCF_002251295.1"""
"""GCF_021531895.1"""
"""GCA_004557565.1"""
…
"""GCF_019052365.1"""
"""GCF_009696175.1"""
"""GCA_004552595.1"""


In [14]:
contig_xx = []
for row in df3.iter_rows(named=True):
    acc = row['acc']
    species = row['gtdb_taxonomy'].split(';')[-1]

    filename = f'PRJEB88111.nanopore/reps/{acc}_genomic.fna.gz'
    for record in screed.open(filename):
        contig_xx.append(dict(contig=record.name.split(' ')[0],
                              species=species,
                              acc=acc))

contig_df = pl.DataFrame(contig_xx)


In [15]:
contig_df

contig,species,acc
str,str,str
"""UREV01000001.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000002.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000003.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000004.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000005.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
…,…,…
"""FR882205.1""","""s__Cryptobacteroides sp0004349…","""GCA_000434935.1"""
"""FR882206.1""","""s__Cryptobacteroides sp0004349…","""GCA_000434935.1"""
"""FR882207.1""","""s__Cryptobacteroides sp0004349…","""GCA_000434935.1"""


In [17]:
contig_df.write_csv('PRJEB88111.nanopore/reps/contigs_to_species.csv')